In [1]:
import pandas as pd

file_path = "https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz"

data = pd.read_csv(file_path, sep="\t", nrows=10000, low_memory=False)

data.head()


,code,url,creator,created_t,created_datetime,last_modified_t,last_modified_datetime,last_modified_by,last_updated_t,last_updated_datetime,...,water-hardness_100g,choline_100g,phylloquinone_100g,beta-glucan_100g,inositol_100g,carnitine_100g,sulphate_100g,nitrate_100g,acidity_100g,carbohydrates-total_100g
0,54,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1582569031,2020-02-24T18:30:31Z,1733085204,2024-12-01T20:33:24Z,NaN,1740205422,2025-02-22T06:23:42Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,63,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1673620307,2023-01-13T14:31:47Z,1750061386,2025-06-16T08:09:46Z,bodysupport,1750061386,2025-06-16T08:09:46Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,114,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1580066482,2020-01-26T19:21:22Z,1751035658,2025-06-27T14:47:38Z,teolemon,1751035658,2025-06-27T14:47:38Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,105,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1572117743,2019-10-26T19:22:23Z,1738073570,2025-01-28T14:12:50Z,NaN,1743653496,2025-04-03T04:11:36Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1722606455,2024-08-02T13:47:35Z,1749171851,2025-06-06T01:04:11Z,altroconsumo,1749171851,2025-06-06T01:04:11Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
df = data.copy()

In [3]:
df = data.dropna(subset=['code']) # Remove rows with missing 'code' values
df = df.sort_values('last_modified_t').drop_duplicates(subset='code', keep='last') # Remove duplicate rows based on 'code' and keep the last modified entry

In [4]:
# Sélection des colonnes finissant par '_100g'
cols_100g = [col for col in df.columns if col.endswith('_100g')]

# Remplacement des valeurs hors intervalle par NaN (sans affichage)
for col in cols_100g:
    numeric_values = pd.to_numeric(df[col], errors='coerce')
    mask = numeric_values.notna() & ~numeric_values.between(0, 100)
    df.loc[mask, col] = float('nan')

In [5]:
# Imputation de salt_100g si manquant et sodium_100g présent
if 'salt_100g' in df.columns and 'sodium_100g' in df.columns:
    salt_na = df['salt_100g'].isna()
    sodium_present = df['sodium_100g'].notna()
    mask = salt_na & sodium_present
    df.loc[mask, 'salt_100g'] = pd.to_numeric(df.loc[mask, 'sodium_100g'], errors='coerce') * 2.54 # salt = sodium * 2.54

In [6]:
# Correction des incohérences : sugars ≤ carbohydrates et saturated_fat ≤ fat (inversion uniquement)
import numpy as np

# sugars_100g ≤ carbohydrates_100g
if 'sugars_100g' in df.columns and 'carbohydrates_100g' in df.columns:
    sugars = pd.to_numeric(df['sugars_100g'], errors='coerce')
    carbs = pd.to_numeric(df['carbohydrates_100g'], errors='coerce')
    mask = sugars > carbs
    for idx in df[mask].index:
        # Inversion des valeurs
        df.at[idx, 'sugars_100g'], df.at[idx, 'carbohydrates_100g'] = carbs[idx], sugars[idx]

# saturated_fat_100g ≤ fat_100g
if 'saturated_fat_100g' in df.columns and 'fat_100g' in df.columns:
    sat_fat = pd.to_numeric(df['saturated_fat_100g'], errors='coerce')
    fat = pd.to_numeric(df['fat_100g'], errors='coerce')
    mask = sat_fat > fat
    for idx in df[mask].index:
        # Inversion des valeurs
        df.at[idx, 'saturated_fat_100g'], df.at[idx, 'fat_100g'] = fat[idx], sat_fat[idx]

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute  import SimpleImputer, IterativeImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier

num_ns_columns   = ["energy_100g","sugars_100g","saturated-fat_100g",
                    "salt_100g","fiber_100g","proteins_100g","fruits-vegetables-nuts_100g"]
num_other        = ["cholesterol_100g","trans-fat_100g"]
cat_low          = ["main_category"]
cat_high         = ["brands"]

# 1. Boolean NA masks (quick and explicit)
for col in num_ns_columns + num_other:
    df[f"is_na_{col}"] = df[col].isna().astype("int8")

# 2. Imputers (fit on TRAIN only!)
imputer_ns   = SimpleImputer(strategy="median")       # but we will NOT apply to NS cols
imputer_num  = IterativeImputer(random_state=0, sample_posterior=True,
                                max_iter=10, initial_strategy="median")

# 3. Column transformer
preproc = ColumnTransformer([
    # keep NS columns as-is (LightGBM can handle NaN); scale if you want
    ("num_ns",   "passthrough", num_ns_columns),

    # other numeric → iterative imputer
    ("num_imp",  Pipeline([("imp", imputer_num)]), num_other),

    # low-card cat
    ("cat_low",  OneHotEncoder(handle_unknown="ignore"), cat_low),

    # high-card cat → target encoding with category_encoders (example)
    ("cat_high", "passthrough", cat_high),  # placeholder; encode later with CatBoostEncoder
], remainder="drop")

# 4. Final pipeline
pipe = Pipeline([
    ("prep", preproc),
    ("clf",  LGBMClassifier(num_leaves=63,
                            class_weight="balanced",
                            n_estimators=400,
                            random_state=42))
])
